In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
model = AutoModel.from_pretrained("microsoft/codebert-base")

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [4]:
code_snippet = """
def add(a, b):
    return a + b
"""

inputs = tokenizer(code_snippet, return_tensors="pt", padding=True, truncation=True)

In [6]:
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run model inference without gradient calculation
with torch.no_grad():
    outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
cls_embedding = last_hidden_states[:, 0, :]

print("CLS embedding shape:", cls_embedding.shape)  # (1, hidden_size), e.g. 768
print(cls_embedding)

CLS embedding shape: torch.Size([1, 768])
tensor([[-1.2060e-01,  1.8082e-01, -4.2081e-02,  7.7381e-02,  7.2140e-02,
         -3.1037e-01,  1.1595e-01,  1.1557e-01,  4.3710e-01, -3.5609e-01,
          1.0073e-01,  1.4747e-01, -2.3316e-01,  5.6936e-02,  4.3202e-01,
         -9.5920e-02, -6.9467e-02,  3.5346e-02, -8.7260e-02,  1.9044e-01,
         -1.8205e-01, -2.1051e-01,  3.9318e-01,  2.4575e-01,  2.0589e-01,
          5.5444e-02,  2.7610e-01,  4.4799e-01,  4.1468e-02,  5.1104e-01,
         -2.2399e-01, -1.5891e-01,  1.6973e+00, -2.7293e-01,  4.1451e-01,
         -1.0663e-01, -1.7174e-01,  1.6464e-01,  1.0577e-01,  1.1116e-01,
         -3.8332e-01,  4.3770e-02, -9.0311e-01,  1.1934e-01,  2.2354e-01,
         -9.5926e-02,  7.7093e-03,  3.2172e-01, -6.2785e-02,  1.5497e-01,
          8.5079e-02,  6.0688e-02, -5.9515e-01, -6.4624e-02, -1.0192e-01,
          3.8080e-01, -8.0128e-01,  1.9108e-01, -2.8675e-01, -3.0833e-01,
         -2.2336e-01, -1.6513e-01, -5.0399e-02, -2.1058e-01,  1.3619e+

In [ ]:

import torch
import time
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

def benchmark_sentence_transformer(model_name, batch_size=16, num_batches=10, device='cuda'):
    model = SentenceTransformer(model_name).to(device)
    model.eval()
    
    texts = ["This is a test sentence."] * batch_size

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    # Warm-up
    _ = model.encode(texts, batch_size=batch_size, convert_to_tensor=True, device=device)
    
    start = time.time()
    for _ in range(num_batches):
        with torch.no_grad():
            _ = model.encode(texts, batch_size=batch_size, convert_to_tensor=True, device=device)
    end = time.time()

    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
    avg_time = (end - start) / num_batches

    print(f"[SentenceTransformer: {model_name}]")
    print(f"Peak GPU Memory: {peak_memory:.2f} MB")
    print(f"Avg Inference Time: {avg_time:.4f} sec\n")


def benchmark_transformer_model(model_name, batch_size=16, seq_length=128, num_batches=10, device='cuda'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    
    texts = ["This is a test sentence."] * batch_size
    inputs = tokenizer(texts, padding='max_length', truncation=True, max_length=seq_length, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    # Warm-up
    _ = model(**inputs)

    start = time.time()
    for _ in range(num_batches):
        with torch.no_grad():
            _ = model(**inputs)
    end = time.time()

    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)  # MB
    avg_time = (end - start) / num_batches

    print(f"[Transformers: {model_name}]")
    print(f"Peak GPU Memory: {peak_memory:.2f} MB")
    print(f"Avg Inference Time: {avg_time:.4f} sec\n")


In [6]:
device = "cpu"  #  'cuda' if torch.cuda.is_available() else 'cpu'

# SentenceTransformer models
benchmark_sentence_transformer("BAAI/bge-base-en", batch_size=16, device=device)

# HuggingFace Transformers (like BERT or RoBERTa)
benchmark_transformer_model("microsoft/codebert-base", batch_size=16, device=device)
# benchmark_transformer_model("roberta-base", batch_size=32, device=device)


[Transformers: microsoft/codebert-base]
Peak GPU Memory: 8.12 MB
Avg Inference Time: 3.3725 sec

